# Stage 03 · Experiment Matrix Full

本 notebook 独立运行，只消费 stage 01 导出的 zip 包。


### 初始化运行常量与通用工具
定义 stage 03 的仓库路径、Drive 路径、脚本入口、模型缓存目录和基础工具函数，为 experiment matrix 全量运行做准备。
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：organize

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "03_Experiment_Matrix_Full"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
REPO_ROOT = Path("/content/ceg_wm_workspace")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_project_root"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
SCRIPT_PATH = REPO_ROOT / "scripts" / "03_Experiment_Matrix_Full.py"
ATTESTATION_ENV_PATH = DRIVE_PROJECT_ROOT / "secrets" / "attestation_env.json"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
PERSISTENT_MODEL_CACHE_ROOT = DRIVE_PROJECT_ROOT / "shared_models"
SEMANTIC_MODEL_SOURCE_URLS = {
    "inspyrenet": "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth",
}

def run_checked(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))


### 挂载 Google Drive 并同步代码仓库
挂载 Google Drive、准备输出根目录，并克隆或更新代码仓库，确保 experiment matrix 使用最新仓库内容。

涉及的 Drive 根目录可以先按下面的树状结构理解：

```text
/content/drive/MyDrive/CEG_WM_Outputs_project_root/
├─ secrets/
│  └─ attestation_env.json      [运行密钥文件]
├─ runtime_state/
│  ├─ 01_Paper_Full_Cuda/
│  │  └─ stage 01               [各次运行的状态索引]
│  └─ 03_Experiment_Matrix_Full/
│     └─ stage 03               [各次运行的状态索引]
├─ exports/
│  └─ 01_Paper_Full_Cuda/
│     └─ stage 01               [导出的输入 zip 包]
└─ logs/ 或各 stage 自带日志目录
   └─ 调试日志与命令执行日志
```

In [ ]:
from google.colab import drive


drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
ensure_dir(DRIVE_PROJECT_ROOT)
ensure_dir(HF_HOME)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_url = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    ).stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print_json(
    "repo_binding",
    {
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "config_path": str(CONFIG_PATH),
        "script_path": str(SCRIPT_PATH),
    },
)


### 安装依赖、补齐系统环境并准备模型与共享阈值输入
按 pyproject.toml 安装项目最小运行依赖，补充 InSPyReNet 运行包，设置 Hugging Face 缓存，解析默认配置，下载所需模型资源，并明确 stage 03 只读复用 stage 01 导出的共享阈值工件。

这一阶段主要涉及两类路径：模型缓存路径与后续只读共享阈值的目标位置。可先按下面的树状结构理解：

```text
REPO_ROOT/
├─ huggingface_cache/
│  ├─ hub/
│  ├─ transformers/
│  └─ <MODEL_SNAPSHOT_PATH 对应的模型快照目录>
└─ <mask.semantic_model_path>
   └─ 语义模型权重文件
   
<run_root>/
└─ global_calibrate/
   └─ artifacts/
      └─ thresholds/
         ├─ thresholds_artifact.json          [stage 03 只读复制]
         └─ threshold_metadata_artifact.json  [stage 03 只读复制]
```

模型文件仍落在仓库侧缓存目录；而 stage 03 会把来自 stage 01 的 thresholds 工件复制到当前 run_root/global_calibrate/artifacts/thresholds 下，作为只读共享阈值输入，避免回写 stage 01 正式产物。

In [ ]:
import urllib.request

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)

if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

run_checked([sys.executable, "-m", "pip", "install", "transparent-background"], cwd=REPO_ROOT)

from huggingface_hub import HfApi, snapshot_download
from scripts.notebook_runtime_common import (
    build_directory_digest_summary,
    collect_weight_summary,
    load_yaml_mapping,
    resolve_model_identity,
)

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
MODEL_IDENTITY = resolve_model_identity(CFG_OBJ)
MASK_CFG = CFG_OBJ.get("mask", {}) if isinstance(CFG_OBJ.get("mask"), dict) else {}
WEIGHT_PATH = REPO_ROOT / str(MASK_CFG.get("semantic_model_path", ""))
WEIGHT_SOURCE = str(MASK_CFG.get("semantic_model_source", ""))
WEIGHT_URL = str(os.environ.get("INSPYRENET_MODEL_URL", SEMANTIC_MODEL_SOURCE_URLS.get(WEIGHT_SOURCE, ""))).strip()
PERSISTENT_WEIGHT_PATH = PERSISTENT_MODEL_CACHE_ROOT / "inspyrenet" / "ckpt_base.pth"
FORCE_WEIGHT_DOWNLOAD = str(os.environ.get("INSPYRENET_FORCE_DOWNLOAD", "")).strip().lower() in {"1", "true", "yes"}
ensure_dir(WEIGHT_PATH.parent)
ensure_dir(PERSISTENT_WEIGHT_PATH.parent)

def is_valid_weight_file(path_obj):
    return path_obj.exists() and path_obj.is_file() and path_obj.stat().st_size > 0

def sync_weight_to_repo(source_path, target_path):
    if source_path.resolve() == target_path.resolve():
        return
    ensure_dir(target_path.parent)
    shutil.copy2(source_path, target_path)

HF_ACCESS_SUMMARY = {"status": "unknown", "detail": "not_checked"}
try:
    hf_user = HfApi().whoami()
    HF_ACCESS_SUMMARY = {"status": "authenticated", "detail": hf_user.get("name", "unknown")}
except Exception:
    HF_ACCESS_SUMMARY = {"status": "anonymous", "detail": "anonymous access"}

ATTESTATION_ENV_STATUS = {
    "status": "not_requested",
    "reason": "stage_03_package_only_no_attestation_env_bootstrap",
}

existing_weight_path = None
for candidate_path in [WEIGHT_PATH, PERSISTENT_WEIGHT_PATH]:
    if is_valid_weight_file(candidate_path):
        existing_weight_path = candidate_path
        break

if existing_weight_path is not None and not FORCE_WEIGHT_DOWNLOAD:
    sync_weight_to_repo(existing_weight_path, WEIGHT_PATH)
    if existing_weight_path.resolve() != PERSISTENT_WEIGHT_PATH.resolve():
        shutil.copy2(existing_weight_path, PERSISTENT_WEIGHT_PATH)
else:
    if not WEIGHT_URL:
        raise RuntimeError(f"unsupported semantic_model_source for notebook bootstrap: {WEIGHT_SOURCE}")
    temp_download_path = PERSISTENT_WEIGHT_PATH.with_suffix(PERSISTENT_WEIGHT_PATH.suffix + ".downloading")
    if temp_download_path.exists():
        temp_download_path.unlink()
    urllib.request.urlretrieve(WEIGHT_URL, temp_download_path)
    if not is_valid_weight_file(temp_download_path):
        temp_download_path.unlink(missing_ok=True)
        raise RuntimeError(f"downloaded semantic weight is invalid: {temp_download_path}")
    temp_download_path.replace(PERSISTENT_WEIGHT_PATH)
    sync_weight_to_repo(PERSISTENT_WEIGHT_PATH, WEIGHT_PATH)

MODEL_SNAPSHOT_PATH = snapshot_download(
    repo_id=str(MODEL_IDENTITY["model_id"]),
    revision=None if MODEL_IDENTITY["revision"] == "<absent>" else str(MODEL_IDENTITY["revision"]),
    cache_dir=str(HF_HOME),
)
MODEL_DOWNLOAD_SUMMARY = build_directory_digest_summary(Path(MODEL_SNAPSHOT_PATH))
WEIGHT_DOWNLOAD_SUMMARY = collect_weight_summary(REPO_ROOT, CFG_OBJ)
print_json(
    "stage_role_binding",
    {
        "stage_name": NOTEBOOK_NAME,
        "mainline_reuse": "consume_stage_01_package_only",
        "model_id": MODEL_IDENTITY["model_id"],
        "hf_revision": MODEL_IDENTITY["revision"],
        "hf_access": HF_ACCESS_SUMMARY,
        "semantic_model_path": str(WEIGHT_PATH),
        "semantic_model_cache_path": str(PERSISTENT_WEIGHT_PATH),
        "model_snapshot_path": MODEL_SNAPSHOT_PATH,
        "model_download_policy": "required_for_experiment_matrix_runtime",
        "semantic_model_url": WEIGHT_URL,
        "force_weight_download": FORCE_WEIGHT_DOWNLOAD,
    },
)
print_json("attestation_env_status", ATTESTATION_ENV_STATUS)
print_json("model_download_summary", MODEL_DOWNLOAD_SUMMARY)
print_json("weight_download_summary", WEIGHT_DOWNLOAD_SUMMARY)
print_json(
    "gpu_tool_status",
    {
        "gpu_tool_available": shutil.which("nvidia-smi") is not None,
        "nvidia_smi_path": shutil.which("nvidia-smi") or "<absent>",
    },
)

### 环境预检与 stage 01 输入包门禁
在执行 experiment matrix 脚本前，先检查配置文件、脚本文件、关键模块、HF 模型可访问性、InSPyReNet 权重状态，以及 stage 01 输入包是否可发现。若存在硬性失败项，应先修复再继续执行。

In [ ]:
import socket
import tempfile
import zipfile

from scripts.notebook_runtime_common import discover_stage_packages, select_latest_stage_package
from scripts.workflow_acceptance_common import detect_stage_03_preflight

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

EXPORT_STAGE_ROOT = DRIVE_PROJECT_ROOT / "exports" / "01_Paper_Full_Cuda"
PRECHECK_PACKAGE_CANDIDATES = []
PRECHECK_SELECTED_SOURCE_PACKAGE = None
if SOURCE_PACKAGE_PATH:
    PRECHECK_SELECTED_SOURCE_PACKAGE = Path(SOURCE_PACKAGE_PATH)
else:
    PRECHECK_PACKAGE_CANDIDATES = discover_stage_packages(EXPORT_STAGE_ROOT)
    if PRECHECK_PACKAGE_CANDIDATES:
        precheck_selected_package = select_latest_stage_package(PRECHECK_PACKAGE_CANDIDATES)
        PRECHECK_SELECTED_SOURCE_PACKAGE = Path(str(precheck_selected_package["package_path"]))

PRECHECK_EXTRACTION_ROOT = Path(tempfile.mkdtemp(prefix="stage03_precheck_"))
PRECHECK_SOURCE_PACKAGE_PATH = PRECHECK_SELECTED_SOURCE_PACKAGE or (PRECHECK_EXTRACTION_ROOT / "missing_source_package.zip")
PRECHECK_SOURCE_THRESHOLDS_PATH = PRECHECK_EXTRACTION_ROOT / "artifacts" / "thresholds" / "thresholds_artifact.json"
PRECHECK_SOURCE_THRESHOLDS_DETAIL = "<absent>"
if PRECHECK_SELECTED_SOURCE_PACKAGE is not None and PRECHECK_SELECTED_SOURCE_PACKAGE.exists():
    try:
        with zipfile.ZipFile(PRECHECK_SELECTED_SOURCE_PACKAGE, "r") as archive_file:
            thresholds_member = "artifacts/thresholds/thresholds_artifact.json"
            if thresholds_member in archive_file.namelist():
                archive_file.extract(thresholds_member, path=PRECHECK_EXTRACTION_ROOT)
                PRECHECK_SOURCE_THRESHOLDS_DETAIL = str(PRECHECK_SOURCE_THRESHOLDS_PATH)
            else:
                PRECHECK_SOURCE_THRESHOLDS_DETAIL = "thresholds member missing in source package"
    except Exception as exc:
        PRECHECK_SOURCE_THRESHOLDS_DETAIL = f"{type(exc).__name__}: {exc}"

STAGE_03_PREFLIGHT = detect_stage_03_preflight(
    CONFIG_PATH,
    PRECHECK_SOURCE_PACKAGE_PATH,
    PRECHECK_SOURCE_THRESHOLDS_PATH,
)
record_precheck(
    "stage 03 preflight",
    STAGE_03_PREFLIGHT["ok"],
    json.dumps(STAGE_03_PREFLIGHT, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "CUDA 可用",
    STAGE_03_PREFLIGHT["gpu_tool_available"],
    STAGE_03_PREFLIGHT["nvidia_smi_path"],
)
record_precheck(
    "attestation env 状态（非硬要求）",
    not STAGE_03_PREFLIGHT["missing_attestation_env_vars"],
    ", ".join(STAGE_03_PREFLIGHT["missing_attestation_env_vars"]) or "ok",
)
record_precheck("配置文件存在", CONFIG_PATH.exists(), str(CONFIG_PATH))
record_precheck("主流程脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck(
    "InSPyReNet 权重存在",
    is_valid_weight_file(WEIGHT_PATH),
    f"path={WEIGHT_PATH}, cache_path={PERSISTENT_WEIGHT_PATH}",
)

for module_name in ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors"]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_model_ok = False
hf_model_detail = "not_checked"
try:
    socket.setdefaulttimeout(15)
    model_info = HfApi().model_info(str(MODEL_IDENTITY["model_id"]))
    hf_model_ok = True
    hf_model_detail = f"accessible: {model_info.id}"
except Exception as exc:
    hf_model_ok = False
    hf_model_detail = f"{type(exc).__name__}: {exc}"
record_precheck("HF 模型可访问", hf_model_ok, hf_model_detail)

if SOURCE_PACKAGE_PATH:
    record_precheck("手动输入包存在", PRECHECK_SOURCE_PACKAGE_PATH.exists(), str(PRECHECK_SOURCE_PACKAGE_PATH))
else:
    record_precheck("stage 01 输入包可发现", len(PRECHECK_PACKAGE_CANDIDATES) > 0, str(EXPORT_STAGE_ROOT))
record_precheck(
    "stage 01 thresholds artifact 可发现",
    PRECHECK_SOURCE_THRESHOLDS_PATH.exists(),
    PRECHECK_SOURCE_THRESHOLDS_DETAIL,
)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 20.0, f"free={free_gb:.1f} GB，建议 >= 20 GB")

hard_fail = []
if not STAGE_03_PREFLIGHT["ok"]:
    hard_fail.append({
        "name": "stage 03 preflight",
        "detail": json.dumps(STAGE_03_PREFLIGHT, ensure_ascii=False, sort_keys=True),
    })

print_json(
    "environment_precheck",
    {
        "items": PRECHECK_RESULTS,
        "stage_03_preflight": STAGE_03_PREFLIGHT,
        "hard_fail": hard_fail,
    },
)

if hard_fail:
    raise RuntimeError(f"environment precheck failed: {hard_fail}")

### 选择 stage 01 输入包并执行 experiment matrix 脚本
自动发现或手动指定 stage 01 导出的 zip 包，随后调用 scripts/03_Experiment_Matrix_Full.py，并读取本次 stage 03 的 summary 与 manifest。

这里最关键的是“输入包位置”和“当前 stage 状态索引”两部分：

```text
DRIVE_PROJECT_ROOT/
├─ exports/
│  └─ 01_Paper_Full_Cuda/
│     └─ <stage_01_package>.zip
└─ runtime_state/
   └─ 03_Experiment_Matrix_Full/
      └─ <stage_run_id>/
         └─ stage_summary.json
            ├─ stage_manifest_path
            ├─ package_manifest_path
            ├─ run_root
            ├─ log_root
            └─ export_root
```

一端绑定 stage 01 的导出包，另一端把 stage 03 本次运行的所有正式输出路径统一登记到 stage_summary.json，供后续校验与诊断反查。

In [ ]:
from pathlib import Path

from scripts.notebook_runtime_common import (
    discover_stage_packages,
    make_stage_run_id,
    select_latest_stage_package,
)

if SOURCE_PACKAGE_PATH:
    SOURCE_PACKAGE = Path(SOURCE_PACKAGE_PATH)
    PACKAGE_CANDIDATES = [
        {
            "package_path": str(SOURCE_PACKAGE),
            "selection_reason": "manual SOURCE_PACKAGE_PATH override",
        }
    ]
    SELECTED_PACKAGE = dict(PACKAGE_CANDIDATES[0])
else:
    EXPORT_STAGE_ROOT = DRIVE_PROJECT_ROOT / "exports" / "01_Paper_Full_Cuda"
    PACKAGE_CANDIDATES = discover_stage_packages(EXPORT_STAGE_ROOT)
    print_json(
        "package_candidates",
        [
            {
                "package_path": item.get("package_path"),
                "package_created_at": item.get("package_created_at"),
                "stage_run_id": item.get("stage_run_id"),
                "validation_error": item.get("validation_error"),
            }
            for item in PACKAGE_CANDIDATES
        ],
    )
    SELECTED_PACKAGE = select_latest_stage_package(PACKAGE_CANDIDATES)
    SOURCE_PACKAGE = Path(str(SELECTED_PACKAGE["package_path"]))

print_json(
    "selected_package",
    {
        "package_path": str(SOURCE_PACKAGE),
        "selection_reason": SELECTED_PACKAGE.get("selection_reason", "manual SOURCE_PACKAGE_PATH override"),
    },
)
STAGE_RUN_ID = make_stage_run_id(NOTEBOOK_NAME)
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--config",
    str(CONFIG_PATH),
    "--source-package",
    str(SOURCE_PACKAGE),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--stage-run-id",
    STAGE_RUN_ID,
]
run_checked(COMMAND, cwd=REPO_ROOT, env=os.environ.copy())

SUMMARY_PATH = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / STAGE_RUN_ID / "stage_summary.json"
STAGE_SUMMARY = load_json(SUMMARY_PATH)
STAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["stage_manifest_path"]))
PACKAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["package_manifest_path"]))
SOURCE_STAGE_MANIFEST = load_json(Path(STAGE_MANIFEST["source_stage_manifest_path"]))
SOURCE_PACKAGE_MANIFEST = load_json(Path(STAGE_MANIFEST["source_package_manifest_path"]))
print_json("stage_summary", STAGE_SUMMARY)
print_json("stage_manifest", STAGE_MANIFEST)
print_json("package_manifest", PACKAGE_MANIFEST)


### 校验 lineage、共享阈值与 experiment matrix 产物
验证来源包 sha256、source thresholds、grid summary、grid manifest、aggregate report 和 workflow summary 是否完整，以确保 experiment matrix 输出闭环成立。

重点校验的是来源链条、只读共享阈值和 stage 03 自身产物目录是否完整，可按下面的树状结构理解：

```text
<run_root>/
├─ global_calibrate/
│  └─ artifacts/
│     └─ thresholds/
│        ├─ thresholds_artifact.json          [新增，来自 stage 01]
│        └─ threshold_metadata_artifact.json  [新增，来自 stage 01]
├─ artifacts/
│  ├─ grid_summary.json                       [新增]
│  ├─ grid_manifest.json                      [新增]
│  ├─ aggregate_report.json                   [新增]
│  ├─ workflow_summary.json                   [新增]
│  ├─ run_closure.json                        [新增或补写]
│  └─ stage_manifest.json                     [新增]
└─ source lineage
   ├─ source_package_path                     [来自 stage 01]
   ├─ source_stage_manifest_path              [来自 stage 01]
   ├─ source_package_manifest_path            [来自 stage 01]
   └─ source_thresholds_artifact_path         [来自 stage 01]
```

要确认的不只是文件存在，还包括：输入包哈希一致、source_stage_name 确实指向 stage 01、只读共享阈值已经正确复制到当前 run_root，以及 experiment matrix 的 grid 和 aggregate 结果都已完整落盘。

In [ ]:
from scripts.notebook_runtime_common import collect_missing_file_entries, compute_file_sha256

SOURCE_PACKAGE_ACTUAL_PATH = Path(STAGE_MANIFEST["source_package_path"])
SOURCE_PACKAGE_ACTUAL_SHA256 = compute_file_sha256(SOURCE_PACKAGE_ACTUAL_PATH)
if SOURCE_PACKAGE_ACTUAL_SHA256 != STAGE_MANIFEST["source_package_sha256"]:
    raise RuntimeError(
        f"source package sha256 mismatch: manifest={STAGE_MANIFEST['source_package_sha256']} actual={SOURCE_PACKAGE_ACTUAL_SHA256}"
    )
if SOURCE_STAGE_MANIFEST.get("stage_name") != "01_Paper_Full_Cuda":
    raise RuntimeError(f"unexpected source stage_name: {SOURCE_STAGE_MANIFEST.get('stage_name')}")
LINEAGE_FIELDS = [
    "source_stage_run_id",
    "source_stage_manifest_path",
    "source_package_manifest_path",
    "source_runtime_config_snapshot_path",
    "source_prompt_snapshot_path",
    "source_thresholds_artifact_path",
]
for field_name in LINEAGE_FIELDS:
    if not STAGE_MANIFEST.get(field_name):
        raise RuntimeError(f"missing required lineage field: {field_name}")
REQUIRED_STAGE_FILES = {
    "stage_manifest": Path(STAGE_SUMMARY["stage_manifest_path"]),
    "package_manifest": Path(STAGE_SUMMARY["package_manifest_path"]),
    "source_package": SOURCE_PACKAGE_ACTUAL_PATH,
    "source_stage_manifest": Path(STAGE_MANIFEST["source_stage_manifest_path"]),
    "source_package_manifest": Path(STAGE_MANIFEST["source_package_manifest_path"]),
    "source_thresholds_artifact": Path(STAGE_MANIFEST["source_thresholds_artifact_path"]),
    "grid_summary": Path(STAGE_MANIFEST["evaluation_report_path"]).parent / "grid_summary.json",
    "grid_manifest": Path(STAGE_MANIFEST["evaluation_report_path"]).parent / "grid_manifest.json",
    "aggregate_report": Path(STAGE_MANIFEST["evaluation_report_path"]),
    "workflow_summary": Path(STAGE_MANIFEST["workflow_summary_path"]),
}
MISSING_STAGE_FILES = collect_missing_file_entries(REQUIRED_STAGE_FILES)
if MISSING_STAGE_FILES:
    raise FileNotFoundError(f"stage 03 validation failed, missing files: {MISSING_STAGE_FILES}")
VALIDATION_RESULT = {
    "stage_name": NOTEBOOK_NAME,
    "stage_run_id": STAGE_RUN_ID,
    "source_stage_run_id": STAGE_MANIFEST["source_stage_run_id"],
    "source_package_sha256": SOURCE_PACKAGE_ACTUAL_SHA256,
    "missing_files": MISSING_STAGE_FILES,
    "status": "ok",
}
print_json("validation_result", VALIDATION_RESULT)


### 汇总 lineage 与实验矩阵诊断信息
汇总 source package、source thresholds、当前阶段关键 manifest 字段以及最新日志内容，帮助快速定位 stage 03 的 lineage 关系与运行问题。

诊断时建议优先按下面的目录关系查看：

```text
stage_summary.json
├─ source_package_path       -> stage 01 导出的 zip 包
├─ run_root                  -> 当前 stage 03 实际产物目录
├─ log_root                  -> 当前 stage 03 日志目录
├─ runtime_state_root        -> 当前 notebook 的状态索引目录
└─ export_root               -> 当前 stage 03 导出目录
<log_root>/
└─ *.log
<run_root>/
├─ global_calibrate/artifacts/thresholds/
├─ artifacts/
│  ├─ grid_summary.json
│  ├─ grid_manifest.json
│  ├─ aggregate_report.json
│  └─ workflow_summary.json
└─ 其他 experiment matrix 运行中间目录
```

如果前面步骤失败，优先看 log_root；如果要确认 stage 03 到底新生成了什么，优先看 run_root 和 export_root。

In [ ]:
from scripts.notebook_runtime_common import summarize_manifest_fields, tail_text_file

LOG_ROOT = Path(STAGE_SUMMARY["log_root"])
LOG_FILES = sorted(LOG_ROOT.rglob("*.log"), key=lambda item: item.stat().st_mtime if item.exists() else 0)
LATEST_LOG_PATH = LOG_FILES[-1] if LOG_FILES else None
DIAGNOSTIC_RESULT = {
    "stage_run_id": STAGE_RUN_ID,
    "source_stage_run_id": STAGE_MANIFEST["source_stage_run_id"],
    "source_thresholds_artifact_path": STAGE_MANIFEST["source_thresholds_artifact_path"],
    "source_package_path": STAGE_MANIFEST["source_package_path"],
    "missing_files": VALIDATION_RESULT["missing_files"],
    "latest_log_path": str(LATEST_LOG_PATH) if LATEST_LOG_PATH else "<absent>",
    "latest_log_tail": tail_text_file(LATEST_LOG_PATH, max_lines=20) if LATEST_LOG_PATH else [],
    "lineage_summary": summarize_manifest_fields(
        STAGE_MANIFEST,
        [
            "source_stage_name",
            "source_stage_run_id",
            "source_package_manifest_path",
            "source_package_manifest_digest",
            "source_runtime_config_snapshot_path",
            "source_prompt_snapshot_path",
            "source_thresholds_artifact_path",
        ],
    ),
    "experiment_matrix_summary": summarize_manifest_fields(
        STAGE_MANIFEST,
        [
            "stage_name",
            "stage_run_id",
            "evaluation_report_path",
            "workflow_summary_path",
            "readonly_shared_thresholds_path",
        ],
    ),
    "status": "ok" if not VALIDATION_RESULT["missing_files"] else "diagnostic_required",
}
print_json("diagnostic_result", DIAGNOSTIC_RESULT)


### 最终保存在 Google Drive 中的文件路径
本 notebook 的输入和输出都位于 Google Drive 挂载目录下。下面沿用 stage 01 第 14 个单元的展示方式，并额外标出哪些目录或文件是本阶段新增的。

```text
/content/drive/MyDrive/CEG_WM_Outputs_project_root/
├─ runtime_state/
│  ├─ 01_Paper_Full_Cuda/
│  │  └─ <source_stage_run_id>/
│  │     └─ stage_summary.json                       [来自 stage 01]
│  └─ 03_Experiment_Matrix_Full/
│     └─ <stage_run_id>/
│        └─ stage_summary.json                       [本阶段新增]
├─ exports/
│  ├─ 01_Paper_Full_Cuda/
│  │  └─ <stage_01_package>.zip                      [来自 stage 01，作为本阶段输入]
│  └─ 03_Experiment_Matrix_Full/
│     └─ <stage_03_package>.zip                      [本阶段新增]
├─ <run_root，对应 stage 03 stage_summary.json 中的 run_root>/
│  ├─ global_calibrate/
│  │  └─ artifacts/
│  │     └─ thresholds/
│  │        ├─ thresholds_artifact.json              [本阶段新增，只读复制自 stage 01]
│  │        └─ threshold_metadata_artifact.json      [本阶段新增，只读复制自 stage 01]
│  ├─ artifacts/
│  │  ├─ grid_summary.json                           [本阶段新增]
│  │  ├─ grid_manifest.json                          [本阶段新增]
│  │  ├─ aggregate_report.json                       [本阶段新增]
│  │  ├─ workflow_summary.json                       [本阶段新增]
│  │  ├─ run_closure.json                            [本阶段新增或补写]
│  │  └─ stage_manifest.json                         [本阶段新增]
│  └─ 其他 experiment matrix 运行中间目录              [本阶段新增]
├─ <export_root，对应 stage 03 stage_summary.json 中的 export_root>/
│  ├─ 03_Experiment_Matrix_Full__<stage_run_id>__from__<source_stage_run_id>.zip
│  ├─ package_manifest.json
│  └─ package_index.json
├─ <package_path，对应 stage 03 stage_summary.json 中的 package_path>
│  └─ stage 03 最终 zip 包                           [本阶段新增]
└─ <log_root，对应 stage 03 stage_summary.json 中的 log_root>/
   └─ *.log                                         [本阶段新增]
```

最终 zip 包以 stage 03 自己的矩阵产物为主，不会把 stage 01 的完整 records 或完整 zip 再打进去；但它会额外带入 stage 01 的只读 thresholds 副本和 lineage 快照。zip 内部的核心相对路径如下：

```text
artifacts/grid_summary.json
artifacts/grid_manifest.json
artifacts/aggregate_report.json
artifacts/run_closure.json
artifacts/workflow_summary.json
artifacts/stage_manifest.json
artifacts/package_manifest.json
artifacts/package_index.json
global_calibrate/artifacts/thresholds/thresholds_artifact.json
global_calibrate/artifacts/thresholds/threshold_metadata_artifact.json
runtime_metadata/runtime_config_snapshot.yaml
lineage/source_stage_manifest.json
lineage/source_package_manifest.json
```

其中：
1. 本阶段新增结果是 grid_summary、grid_manifest、aggregate_report、workflow_summary、run_closure。
2. 来自 stage 01 的内容有两类：lineage 快照，以及只读复制进来的 shared thresholds。
3. stage 03 zip 不包含 stage 01 的 embed_record、detect_record、calibration_record、evaluate_record。

建议的查找顺序是：
1. 先打开 stage 03 的 stage_summary.json。
2. 根据其中的 run_root、log_root、export_root、package_path 跳转到真实落盘位置。
3. 如果要核对输入来源，查看 exports/01_Paper_Full_Cuda 下的 source package，以及 runtime_state/01_Paper_Full_Cuda 下对应 source_stage_run_id 的 stage_summary.json。
4. 如果要核对共享阈值绑定，优先查看 run_root/global_calibrate/artifacts/thresholds 与 source_thresholds_artifact_path。
5. 如果只关心本阶段新增结果，优先查看 run_root、export_root、package_path 和 log_root。